# Iris Classification with Gemma LLM - Kaggle P100 Training

This notebook trains a Gemma-3-1b-it model for Iris flower classification using natural language descriptions. It's optimized for Kaggle's P100 GPU environment.

## Setup Instructions
1. Enable GPU in Kaggle (P100)
2. Add your HuggingFace token as a Kaggle secret named 'HF_TOKEN'
3. Upload your iris-pipeline repository files
4. Run all cells to train the model
5. Download the trained model for local use


In [ ]:
# Install required packages
!pip install transformers torch peft trl datasets accelerate bitsandbytes -q
!pip install pandas scikit-learn -q

In [ ]:
import os
import torch
import pandas as pd
import json
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from huggingface_hub import login

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Configuration
BASE_MODEL_ID = "google/gemma-3-1b-it"
OUTPUT_DIR = "iris-gemma-model"
HF_TOKEN = os.getenv('HF_TOKEN')  # Set in Kaggle secrets

# Login to HuggingFace
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ Logged into HuggingFace")
else:
    print("⚠️ No HF token found. Set HF_TOKEN in Kaggle secrets.")

In [ ]:
# Create sample Iris dataset if not available
iris_data = {
    'sepal_length': [5.1, 4.9, 4.7, 4.6, 5.0, 5.4, 4.6, 5.0, 4.4, 4.9,
                     7.0, 6.4, 6.9, 5.5, 6.5, 5.7, 6.3, 4.9, 6.6, 5.2,
                     6.3, 5.8, 7.1, 6.3, 6.5, 7.6, 4.9, 7.3, 6.7, 7.2],
    'sepal_width': [3.5, 3.0, 3.2, 3.1, 3.6, 3.9, 3.4, 3.4, 2.9, 3.1,
                    3.2, 3.2, 3.1, 2.3, 2.8, 2.8, 3.3, 2.4, 2.9, 2.7,
                    3.3, 2.7, 3.0, 2.9, 3.0, 3.0, 2.5, 2.9, 2.5, 3.6],
    'petal_length': [1.4, 1.4, 1.3, 1.5, 1.4, 1.7, 1.4, 1.5, 1.4, 1.5,
                     4.7, 4.5, 4.9, 4.0, 4.6, 4.5, 4.7, 3.3, 4.6, 3.9,
                     6.0, 5.1, 5.9, 5.6, 5.8, 6.6, 4.5, 6.3, 5.8, 6.1],
    'petal_width': [0.2, 0.2, 0.2, 0.2, 0.2, 0.4, 0.3, 0.2, 0.2, 0.1,
                    1.4, 1.5, 1.5, 1.3, 1.5, 1.3, 1.6, 1.0, 1.3, 1.4,
                    2.5, 1.9, 2.1, 1.8, 2.2, 2.1, 1.7, 1.8, 1.8, 2.5],
    'species': ['setosa']*10 + ['versicolor']*10 + ['virginica']*10
}

iris_df = pd.DataFrame(iris_data)
print(f"Created Iris dataset with {len(iris_df)} samples")
print(iris_df.head())

In [ ]:
# Convert to linguistic format
def create_linguistic_prompt(row):
    """Convert iris row to chat format."""
    messages = [
        {
            "role": "system",
            "content": "Classify the flower based on its measurements into one of the following species: [Setosa, Versicolor, Virginica]"
        },
        {
            "role": "user",
            "content": f"Sepal Length: {row['sepal_length']}, Sepal Width: {row['sepal_width']}, Petal Length: {row['petal_length']}, Petal Width: {row['petal_width']}"
        },
        {
            "role": "assistant",
            "content": row['species'].capitalize()
        }
    ]
    return messages

# Create training data
training_data = []
for _, row in iris_df.iterrows():
    messages = create_linguistic_prompt(row)
    # Format as single text for training
    text = f"<system>{messages[0]['content']}</system>\n"
    text += f"<user>{messages[1]['content']}</user>\n"
    text += f"<assistant>{messages[2]['content']}</assistant>"
    training_data.append({"text": text})

# Create dataset
dataset = Dataset.from_list(training_data)
print(f"Created training dataset with {len(dataset)} samples")
print("Sample training text:")
print(dataset[0]['text'])

In [ ]:
# Load the base model
print(f"Loading base model: {BASE_MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# Prepare for training
model = prepare_model_for_kbit_training(model)
print("✅ Model loaded and prepared")

In [ ]:
# Setup LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", 
                   "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ LoRA configuration applied")

In [ ]:
# Training configuration optimized for P100
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,  # Small batch for P100
    gradient_accumulation_steps=8,   # Effective batch size = 16
    warmup_steps=50,
    logging_steps=5,
    save_steps=50,
    evaluation_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=True,  # Use mixed precision
    dataloader_drop_last=True,
    remove_unused_columns=False,
    report_to="none"
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("✅ Training arguments configured")

In [ ]:
# Create trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    data_collator=data_collator,
    tokenizer=tokenizer,
    max_seq_length=512,
    dataset_text_field="text"
)

print("✅ Trainer created")

In [ ]:
# Start training
print("🚀 Starting training...")
trainer.train()
print("✅ Training completed!")

In [ ]:
# Save the model
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

# List saved files
import os
print("\nSaved files:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
# Test the trained model
print("Testing the trained model...")

def test_prediction(sepal_length, sepal_width, petal_length, petal_width):
    """Test a single prediction."""
    messages = [
        {
            "role": "system",
            "content": "Classify the flower based on its measurements into one of the following species: [Setosa, Versicolor, Virginica]"
        },
        {
            "role": "user",
            "content": f"Sepal Length: {sepal_length}, Sepal Width: {sepal_width}, Petal Length: {petal_length}, Petal Width: {petal_width}"
        }
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt")
    
    if torch.cuda.is_available():
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            temperature=0.1,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )
    
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], 
        skip_special_tokens=True
    ).strip()
    
    return response

# Test with known samples
test_cases = [
    (5.1, 3.5, 1.4, 0.2, "setosa"),
    (6.4, 3.2, 4.5, 1.5, "versicolor"),
    (6.3, 3.3, 6.0, 2.5, "virginica")
]

print("\nTest Results:")
for sl, sw, pl, pw, expected in test_cases:
    prediction = test_prediction(sl, sw, pl, pw)
    print(f"Input: {sl}, {sw}, {pl}, {pw}")
    print(f"Expected: {expected}")
    print(f"Predicted: {prediction}")
    print("-" * 50)

In [ ]:
# Create a zip file for download
import shutil

# Create zip file
shutil.make_archive('iris-gemma-model', 'zip', OUTPUT_DIR)
print("\n✅ Model packaged as iris-gemma-model.zip")
print("📥 Download this file to use the model locally!")

# Show file size
import os
size_mb = os.path.getsize('iris-gemma-model.zip') / (1024*1024)
print(f"📦 Package size: {size_mb:.1f} MB")

## Next Steps

1. **Download the model**: Download `iris-gemma-model.zip` from this Kaggle session
2. **Extract locally**: Unzip to `models/iris-gemma-local/` in your project
3. **Use locally**: The model can now be used with `LocalGemmaIrisClassifier`

### Local Usage Example:
```python
from src.gemma_classifier import LocalGemmaIrisClassifier

# Load your trained model
classifier = LocalGemmaIrisClassifier(
    local_model_path="models/iris-gemma-local",
    hf_token="your_token_here"
)

classifier.load_model()
species, confidence = classifier.predict(5.1, 3.5, 1.4, 0.2)
print(f"Prediction: {species} (confidence: {confidence})")
```